# School Scenario Test

## Overview
Testing multi-tenant school system with notes creation and assignment.

## Service Ports
| Service | Port |
|---------|------|
| Role-Permission | 8080 |
| Auth Service | 8081 |
| Tenant Service | 8082 |
| User Profile | 8083 |
| API Gateway | 8084 |
| Notes | 8088 |

---

## Scenario

### Tenants (Schools)
| ID | School Name |
|----|-------------|
| 1 | Euroschool |
| 2 | Ravishankarschool |

### Users
| Name | Role | School | Class/Section | Subjects |
|------|------|--------|---------------|----------|
| Ammu | Student | Euroschool | 4-C | Science, Social Science |
| Priya | Subject Teacher | Euroschool | 4-C | Science, Social Science |
| Principal 1 | Principal | Euroschool | All | All |
| Daksh | Student | Ravishankarschool | 4-D | Science, Social Science |
| Ravi | Subject Teacher | Ravishankarschool | 4-D | Science, Social Science |
| Principal 2 | Principal | Ravishankarschool | All | All |

### Access Rules
| Role | Can Create Notes | Can View Notes |
|------|------------------|----------------|
| Principal | No | All notes in their school |
| Subject Teacher | Yes | Only their assigned class |
| Student | No | Only notes assigned to them |

---

## Setup: Import and Helper Functions

In [59]:
import requests
import json

# Base URLs
ROLE_PERMISSION_URL = "http://localhost:8080"
AUTH_URL = "http://localhost:8081"
TENANT_URL = "http://localhost:8082"
USER_PROFILE_URL = "http://localhost:8083"
API_GATEWAY_URL = "http://localhost:8084"
NOTES_URL = "http://localhost:8088"

# Store created resources for cleanup
created_resources = {
    "tenants": [],
    "users": [],
    "roles": [],
    "grants": [],
    "assignments": [],
    "notes": []
}

def print_response(response, label="Response"):
    print(f"{label} - Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except:
        print(response.text)

print("Setup complete!")

Setup complete!


---

## Step 1: Create Tenants (Schools)

In [60]:
# Get or Create Euroschool tenant
# tenantKey must be lowercase alphanumeric with hyphens (pattern: ^[a-z0-9](?:[a-z0-9-]{1,48}[a-z0-9])$)

# First check if tenant already exists
print("Checking for existing Euroschool tenant...")
response = requests.get(f"{TENANT_URL}/v1/tenants")
euroschool_id = None

if response.status_code == 200:
    for t in response.json():
        if t.get("tenantKey") == "euroschool":
            euroschool_id = t.get("id")
            print(f"Euroschool already exists (ID: {euroschool_id})")
            break

if not euroschool_id:
    # Create new tenant
    response = requests.post(
        f"{TENANT_URL}/v1/tenants",
        headers={"Content-Type": "application/json"},
        json={
            "tenantKey": "euroschool",
            "name": "Euroschool"
        }
    )
    
    print("Creating Euroschool...")
    print_response(response)
    
    if response.status_code in [200, 201]:
        euroschool = response.json()
        euroschool_id = euroschool.get("id")
        created_resources["tenants"].append(euroschool_id)

print(f"\nEuroschool ID: {euroschool_id}")

Checking for existing Euroschool tenant...
Euroschool already exists (ID: 0f66209d-f36f-4567-b24a-3d32a31bce80)

Euroschool ID: 0f66209d-f36f-4567-b24a-3d32a31bce80


In [61]:
# Get or Create Ravishankarschool tenant
# tenantKey must be lowercase alphanumeric with hyphens

# First check if tenant already exists
print("Checking for existing Ravishankarschool tenant...")
response = requests.get(f"{TENANT_URL}/v1/tenants")
ravishankarschool_id = None

if response.status_code == 200:
    for t in response.json():
        if t.get("tenantKey") == "ravishankarschool":
            ravishankarschool_id = t.get("id")
            print(f"Ravishankarschool already exists (ID: {ravishankarschool_id})")
            break

if not ravishankarschool_id:
    # Create new tenant
    response = requests.post(
        f"{TENANT_URL}/v1/tenants",
        headers={"Content-Type": "application/json"},
        json={
            "tenantKey": "ravishankarschool",
            "name": "Ravishankar School"
        }
    )
    
    print("Creating Ravishankarschool...")
    print_response(response)
    
    if response.status_code in [200, 201]:
        ravishankarschool = response.json()
        ravishankarschool_id = ravishankarschool.get("id")
        created_resources["tenants"].append(ravishankarschool_id)

print(f"\nRavishankarschool ID: {ravishankarschool_id}")

Checking for existing Ravishankarschool tenant...
Ravishankarschool already exists (ID: 43e49388-a868-45ae-a45c-7de2f646871d)

Ravishankarschool ID: 43e49388-a868-45ae-a45c-7de2f646871d


In [ ]:
# Verify tenants created - List all tenants
response = requests.get(f"{TENANT_URL}/v1/tenants")

print("Listing all tenants...")
print_response(response)

# Extract tenant IDs for use in subsequent cells
if response.status_code == 200:
    tenants = response.json()
    for t in tenants:
        # Match by tenantKey (lowercase identifier)
        if t.get("tenantKey") == "euroschool":
            euroschool_id = t.get("id")
        elif t.get("tenantKey") == "ravishankarschool":
            ravishankarschool_id = t.get("id")
    
    print(f"\nEuroschool ID: {euroschool_id}")
    print(f"Ravishankarschool ID: {ravishankarschool_id}")

---

## Step 2: Create Roles and Permissions

### 2.1 Create Permissions (Global)

In [62]:
# Get or Create notes permissions
# First, fetch existing permissions to avoid duplicates

print("Checking existing permissions...")
response = requests.get(f"{ROLE_PERMISSION_URL}/permissions")
existing_permissions = {}
if response.status_code == 200:
    for p in response.json():
        # Permission code is typically RESOURCE:ACTION
        code = f"{p.get('resource')}:{p.get('action')}"
        existing_permissions[code] = p.get("id")
    print(f"Found {len(existing_permissions)} existing permission(s)")

permissions_to_create = [
    {"description": "Allow creating new notes", "resource": "NOTE", "action": "CREATE", "active": True},
    {"description": "Allow reading notes", "resource": "NOTE", "action": "READ", "active": True},
    {"description": "Allow updating notes", "resource": "NOTE", "action": "UPDATE", "active": True},
    {"description": "Allow deleting notes", "resource": "NOTE", "action": "DELETE", "active": True},
]

print("\nEnsuring Notes Permissions exist...")
print("-" * 40)

for perm in permissions_to_create:
    code = f"{perm['resource']}:{perm['action']}"
    
    if code in existing_permissions:
        print(f"{code} - Already exists (ID: {existing_permissions[code]})")
    else:
        response = requests.post(
            f"{ROLE_PERMISSION_URL}/permissions",
            headers={"Content-Type": "application/json"},
            json=perm
        )
        if response.status_code in [200, 201]:
            perm_data = response.json()
            existing_permissions[code] = perm_data.get("id")
            print(f"{code} - Created (ID: {perm_data.get('id')})")
        else:
            print(f"{code} - Error: {response.status_code}")

print("-" * 40)
print("Permissions ready!")

Checking existing permissions...
Found 4 existing permission(s)

Ensuring Notes Permissions exist...
----------------------------------------
NOTE:CREATE - Already exists (ID: 6)
NOTE:READ - Already exists (ID: 3)
NOTE:UPDATE - Already exists (ID: 7)
NOTE:DELETE - Already exists (ID: 8)
----------------------------------------
Permissions ready!


### 2.2 Create Roles for Euroschool

In [63]:
# Get or Create Roles for Euroschool
# Make sure euroschool_id is set from Step 1

# First, fetch existing roles for this tenant
print(f"Checking existing roles for Euroschool (ID: {euroschool_id})...")
response = requests.get(f"{ROLE_PERMISSION_URL}/tenants/{euroschool_id}/roles")
existing_roles = {}
if response.status_code == 200:
    for r in response.json():
        existing_roles[r.get("name")] = r.get("id")
    print(f"Found {len(existing_roles)} existing role(s): {list(existing_roles.keys())}")

euroschool_roles = [
    {"name": "PRINCIPAL", "description": "School Principal - can view all notes", "active": True},
    {"name": "SUBJECT_TEACHER", "description": "Subject Teacher - can create/view notes for assigned class", "active": True},
    {"name": "STUDENT", "description": "Student - can view assigned notes only", "active": True},
]

print(f"\nEnsuring roles exist for Euroschool...")
print("-" * 40)

euroschool_role_ids = {}

for role in euroschool_roles:
    role_name = role["name"]
    
    if role_name in existing_roles:
        # Role already exists, use existing ID
        euroschool_role_ids[role_name] = existing_roles[role_name]
        print(f"{role_name} - Already exists (ID: {existing_roles[role_name]})")
    else:
        # Create new role
        response = requests.post(
            f"{ROLE_PERMISSION_URL}/tenants/{euroschool_id}/roles",
            headers={"Content-Type": "application/json"},
            json=role
        )
        if response.status_code in [200, 201]:
            role_data = response.json()
            euroschool_role_ids[role_name] = role_data.get("id")
            created_resources["roles"].append({"tenant_id": euroschool_id, "role_id": role_data.get("id")})
            print(f"{role_name} - Created (ID: {role_data.get('id')})")
        else:
            print(f"{role_name} - Error: {response.text}")

print("-" * 40)
print(f"Euroschool Role IDs: {euroschool_role_ids}")

Checking existing roles for Euroschool (ID: 0f66209d-f36f-4567-b24a-3d32a31bce80)...
Found 3 existing role(s): ['PRINCIPAL', 'SUBJECT_TEACHER', 'STUDENT']

Ensuring roles exist for Euroschool...
----------------------------------------
PRINCIPAL - Already exists (ID: 3)
SUBJECT_TEACHER - Already exists (ID: 4)
STUDENT - Already exists (ID: 5)
----------------------------------------
Euroschool Role IDs: {'PRINCIPAL': 3, 'SUBJECT_TEACHER': 4, 'STUDENT': 5}


### 2.3 Create Roles for Ravishankarschool

In [64]:
# Get or Create Roles for Ravishankarschool
# Make sure ravishankarschool_id is set from Step 1

# First, fetch existing roles for this tenant
print(f"Checking existing roles for Ravishankarschool (ID: {ravishankarschool_id})...")
response = requests.get(f"{ROLE_PERMISSION_URL}/tenants/{ravishankarschool_id}/roles")
existing_roles = {}
if response.status_code == 200:
    for r in response.json():
        existing_roles[r.get("name")] = r.get("id")
    print(f"Found {len(existing_roles)} existing role(s): {list(existing_roles.keys())}")

ravishankar_roles = [
    {"name": "PRINCIPAL", "description": "School Principal - can view all notes", "active": True},
    {"name": "SUBJECT_TEACHER", "description": "Subject Teacher - can create/view notes for assigned class", "active": True},
    {"name": "STUDENT", "description": "Student - can view assigned notes only", "active": True},
]

print(f"\nEnsuring roles exist for Ravishankarschool...")
print("-" * 40)

ravishankar_role_ids = {}

for role in ravishankar_roles:
    role_name = role["name"]
    
    if role_name in existing_roles:
        # Role already exists, use existing ID
        ravishankar_role_ids[role_name] = existing_roles[role_name]
        print(f"{role_name} - Already exists (ID: {existing_roles[role_name]})")
    else:
        # Create new role
        response = requests.post(
            f"{ROLE_PERMISSION_URL}/tenants/{ravishankarschool_id}/roles",
            headers={"Content-Type": "application/json"},
            json=role
        )
        if response.status_code in [200, 201]:
            role_data = response.json()
            ravishankar_role_ids[role_name] = role_data.get("id")
            created_resources["roles"].append({"tenant_id": ravishankarschool_id, "role_id": role_data.get("id")})
            print(f"{role_name} - Created (ID: {role_data.get('id')})")
        else:
            print(f"{role_name} - Error: {response.text}")

print("-" * 40)
print(f"Ravishankarschool Role IDs: {ravishankar_role_ids}")

Checking existing roles for Ravishankarschool (ID: 43e49388-a868-45ae-a45c-7de2f646871d)...
Found 3 existing role(s): ['PRINCIPAL', 'SUBJECT_TEACHER', 'STUDENT']

Ensuring roles exist for Ravishankarschool...
----------------------------------------
PRINCIPAL - Already exists (ID: 6)
SUBJECT_TEACHER - Already exists (ID: 7)
STUDENT - Already exists (ID: 8)
----------------------------------------
Ravishankarschool Role IDs: {'PRINCIPAL': 6, 'SUBJECT_TEACHER': 7, 'STUDENT': 8}


### 2.4 Assign Permissions to Roles - Euroschool

In [65]:
# Assign permissions to Euroschool roles

# Permission assignments:
# - PRINCIPAL: NOTE:READ with TENANT scope (can see all notes in school)
# - SUBJECT_TEACHER: NOTE:CREATE, NOTE:READ, NOTE:UPDATE, NOTE:DELETE with CLASS scope
# - STUDENT: NOTE:READ with OWN scope (only assigned notes)

print(f"Assigning permissions to Euroschool roles...")
print("-" * 40)

# Principal - can read all notes in tenant
if "PRINCIPAL" in euroschool_role_ids:
    response = requests.post(
        f"{ROLE_PERMISSION_URL}/tenants/{euroschool_id}/roles/{euroschool_role_ids['PRINCIPAL']}/grants",
        headers={"Content-Type": "application/json"},
        json={"permissionCode": "NOTE:READ", "scopeCode": "TENANT"}
    )
    print(f"PRINCIPAL + NOTE:READ (TENANT) - Status: {response.status_code}")

# Subject Teacher - full CRUD on notes for their class
if "SUBJECT_TEACHER" in euroschool_role_ids:
    teacher_permissions = ["NOTE:CREATE", "NOTE:READ", "NOTE:UPDATE", "NOTE:DELETE"]
    for perm in teacher_permissions:
        response = requests.post(
            f"{ROLE_PERMISSION_URL}/tenants/{euroschool_id}/roles/{euroschool_role_ids['SUBJECT_TEACHER']}/grants",
            headers={"Content-Type": "application/json"},
            json={"permissionCode": perm, "scopeCode": "CLASS"}
        )
        print(f"SUBJECT_TEACHER + {perm} (CLASS) - Status: {response.status_code}")

# Student - can only read notes assigned to them
if "STUDENT" in euroschool_role_ids:
    response = requests.post(
        f"{ROLE_PERMISSION_URL}/tenants/{euroschool_id}/roles/{euroschool_role_ids['STUDENT']}/grants",
        headers={"Content-Type": "application/json"},
        json={"permissionCode": "NOTE:READ", "scopeCode": "OWN"}
    )
    print(f"STUDENT + NOTE:READ (OWN) - Status: {response.status_code}")

print("-" * 40)
print("Euroschool permissions assigned!")

Assigning permissions to Euroschool roles...
----------------------------------------
PRINCIPAL + NOTE:READ (TENANT) - Status: 200
SUBJECT_TEACHER + NOTE:CREATE (CLASS) - Status: 200
SUBJECT_TEACHER + NOTE:READ (CLASS) - Status: 200
SUBJECT_TEACHER + NOTE:UPDATE (CLASS) - Status: 200
SUBJECT_TEACHER + NOTE:DELETE (CLASS) - Status: 200
STUDENT + NOTE:READ (OWN) - Status: 200
----------------------------------------
Euroschool permissions assigned!


### 2.5 Assign Permissions to Roles - Ravishankarschool

In [66]:
# Assign permissions to Ravishankarschool roles (same pattern as Euroschool)

print(f"Assigning permissions to Ravishankarschool roles...")
print("-" * 40)

# Principal - can read all notes in tenant
if "PRINCIPAL" in ravishankar_role_ids:
    response = requests.post(
        f"{ROLE_PERMISSION_URL}/tenants/{ravishankarschool_id}/roles/{ravishankar_role_ids['PRINCIPAL']}/grants",
        headers={"Content-Type": "application/json"},
        json={"permissionCode": "NOTE:READ", "scopeCode": "TENANT"}
    )
    print(f"PRINCIPAL + NOTE:READ (TENANT) - Status: {response.status_code}")

# Subject Teacher - full CRUD on notes for their class
if "SUBJECT_TEACHER" in ravishankar_role_ids:
    teacher_permissions = ["NOTE:CREATE", "NOTE:READ", "NOTE:UPDATE", "NOTE:DELETE"]
    for perm in teacher_permissions:
        response = requests.post(
            f"{ROLE_PERMISSION_URL}/tenants/{ravishankarschool_id}/roles/{ravishankar_role_ids['SUBJECT_TEACHER']}/grants",
            headers={"Content-Type": "application/json"},
            json={"permissionCode": perm, "scopeCode": "CLASS"}
        )
        print(f"SUBJECT_TEACHER + {perm} (CLASS) - Status: {response.status_code}")

# Student - can only read notes assigned to them
if "STUDENT" in ravishankar_role_ids:
    response = requests.post(
        f"{ROLE_PERMISSION_URL}/tenants/{ravishankarschool_id}/roles/{ravishankar_role_ids['STUDENT']}/grants",
        headers={"Content-Type": "application/json"},
        json={"permissionCode": "NOTE:READ", "scopeCode": "OWN"}
    )
    print(f"STUDENT + NOTE:READ (OWN) - Status: {response.status_code}")

print("-" * 40)
print("Ravishankarschool permissions assigned!")

Assigning permissions to Ravishankarschool roles...
----------------------------------------
PRINCIPAL + NOTE:READ (TENANT) - Status: 200
SUBJECT_TEACHER + NOTE:CREATE (CLASS) - Status: 200
SUBJECT_TEACHER + NOTE:READ (CLASS) - Status: 200
SUBJECT_TEACHER + NOTE:UPDATE (CLASS) - Status: 200
SUBJECT_TEACHER + NOTE:DELETE (CLASS) - Status: 200
STUDENT + NOTE:READ (OWN) - Status: 200
----------------------------------------
Ravishankarschool permissions assigned!


---

## Step 3: Create Users

### 3.1 Create Euroschool Users

In [67]:
# Create users for Euroschool
# Users: Ammu (Student), Priya (Teacher), Principal1

euroschool_users = [
    {"email": "ammu@euroschool.com", "password": "ammu123", "name": "Ammu", "role": "STUDENT"},
    {"email": "priya@euroschool.com", "password": "priya123", "name": "Priya", "role": "SUBJECT_TEACHER"},
    {"email": "principal@euroschool.com", "password": "principal123", "name": "Principal1", "role": "PRINCIPAL"},
]

print(f"Creating users for Euroschool (Tenant ID: {euroschool_id})...")
print("-" * 40)

euroschool_user_data = {}

for user in euroschool_users:
    response = requests.post(
        f"{AUTH_URL}/auth/signup",
        headers={"Content-Type": "application/json"},
        json={
            "tenantId": str(euroschool_id),
            "email": user["email"],
            "password": user["password"],
            "name": user["name"],
            "joinMethod": "SELF_SIGNUP"
        }
    )
    
    if response.status_code in [200, 201]:
        user_data = response.json()
        euroschool_user_data[user["name"]] = {
            "userId": user_data.get("userId"),
            "accessToken": user_data.get("accessToken"),
            "role": user["role"]
        }
        created_resources["users"].append({"tenant_id": euroschool_id, "user_id": user_data.get("userId")})
        print(f"{user['name']} ({user['role']}) - Created (ID: {user_data.get('userId')})")
    else:
        print(f"{user['name']} - Error: {response.text}")

print("-" * 40)
print(f"Euroschool Users: {json.dumps(euroschool_user_data, indent=2)}")

Creating users for Euroschool (Tenant ID: 0f66209d-f36f-4567-b24a-3d32a31bce80)...
----------------------------------------
Ammu (STUDENT) - Created (ID: 60f3c705-7fcb-43a3-b5ab-4b8cbfecbc43)
Priya (SUBJECT_TEACHER) - Created (ID: cb23f11f-0294-4ccb-9eaa-77c8779461f2)
Principal1 (PRINCIPAL) - Created (ID: 6846ea99-d0fa-430f-a323-4775b67a5011)
----------------------------------------
Euroschool Users: {
  "Ammu": {
    "userId": "60f3c705-7fcb-43a3-b5ab-4b8cbfecbc43",
    "accessToken": "466dc3bc-8775-4e69-97bb-7f37542a76a2",
    "role": "STUDENT"
  },
  "Priya": {
    "userId": "cb23f11f-0294-4ccb-9eaa-77c8779461f2",
    "accessToken": "55467717-ef6f-42d9-bc0f-2bc91670c4d8",
    "role": "SUBJECT_TEACHER"
  },
  "Principal1": {
    "userId": "6846ea99-d0fa-430f-a323-4775b67a5011",
    "accessToken": "5a0023a2-c90f-40c0-b3cf-5f05a4bd53d7",
    "role": "PRINCIPAL"
  }
}


### 3.2 Create Ravishankarschool Users

In [68]:
# Create users for Ravishankarschool
# Users: Daksh (Student), Ravi (Teacher), Principal2

ravishankar_users = [
    {"email": "daksh@ravishankarschool.com", "password": "daksh123", "name": "Daksh", "role": "STUDENT"},
    {"email": "ravi@ravishankarschool.com", "password": "ravi123", "name": "Ravi", "role": "SUBJECT_TEACHER"},
    {"email": "principal@ravishankarschool.com", "password": "principal123", "name": "Principal2", "role": "PRINCIPAL"},
]

print(f"Creating users for Ravishankarschool (Tenant ID: {ravishankarschool_id})...")
print("-" * 40)

ravishankar_user_data = {}

for user in ravishankar_users:
    response = requests.post(
        f"{AUTH_URL}/auth/signup",
        headers={"Content-Type": "application/json"},
        json={
            "tenantId": str(ravishankarschool_id),
            "email": user["email"],
            "password": user["password"],
            "name": user["name"],
            "joinMethod": "SELF_SIGNUP"
        }
    )
    
    if response.status_code in [200, 201]:
        user_data = response.json()
        ravishankar_user_data[user["name"]] = {
            "userId": user_data.get("userId"),
            "accessToken": user_data.get("accessToken"),
            "role": user["role"]
        }
        created_resources["users"].append({"tenant_id": ravishankarschool_id, "user_id": user_data.get("userId")})
        print(f"{user['name']} ({user['role']}) - Created (ID: {user_data.get('userId')})")
    else:
        print(f"{user['name']} - Error: {response.text}")

print("-" * 40)
print(f"Ravishankarschool Users: {json.dumps(ravishankar_user_data, indent=2)}")

Creating users for Ravishankarschool (Tenant ID: 43e49388-a868-45ae-a45c-7de2f646871d)...
----------------------------------------
Daksh (STUDENT) - Created (ID: f13d09b7-4645-4fdb-9322-7cc6ecda9d0e)
Ravi (SUBJECT_TEACHER) - Created (ID: 1b123b44-12d9-4102-a756-bdebb2ac1e12)
Principal2 (PRINCIPAL) - Created (ID: f56af500-e940-4b0a-8a9f-5a2d50302f74)
----------------------------------------
Ravishankarschool Users: {
  "Daksh": {
    "userId": "f13d09b7-4645-4fdb-9322-7cc6ecda9d0e",
    "accessToken": "c0e93c0b-dbea-4532-85db-f0904f98bc26",
    "role": "STUDENT"
  },
  "Ravi": {
    "userId": "1b123b44-12d9-4102-a756-bdebb2ac1e12",
    "accessToken": "d9937913-56ac-4d40-bbbe-d66ea3bb9030",
    "role": "SUBJECT_TEACHER"
  },
  "Principal2": {
    "userId": "f56af500-e940-4b0a-8a9f-5a2d50302f74",
    "accessToken": "821b0d34-2bce-4927-a163-de1d6e7aa4f1",
    "role": "PRINCIPAL"
  }
}


---

## Step 4: Assign Roles to Users

### 4.1 Assign Roles to Euroschool Users

In [69]:
# Assign roles to Euroschool users

print(f"Assigning roles to Euroschool users...")
print("-" * 40)

for user_name, user_info in euroschool_user_data.items():
    role_name = user_info["role"]
    user_id = user_info["userId"]
    role_id = euroschool_role_ids.get(role_name)
    
    if not role_id:
        print(f"{user_name} - Role {role_name} not found!")
        continue
    
    # Determine scope based on role
    if role_name == "PRINCIPAL":
        scope_type = "TENANT"
        scope_id = None
    elif role_name == "SUBJECT_TEACHER":
        scope_type = "CLASS"
        scope_id = "4-C"  # Priya teaches Class 4-C
    else:  # STUDENT
        scope_type = "CLASS"
        scope_id = "4-C"  # Ammu is in Class 4-C
    
    payload = {
        "roleId": role_id,
        "scopeType": scope_type,
        "scopeId": scope_id,  # Must always be included, even when None
        "status": "ACTIVE"
    }
    
    response = requests.post(
        f"{ROLE_PERMISSION_URL}/tenants/{euroschool_id}/users/{user_id}/roles",
        headers={"Content-Type": "application/json"},
        json=payload
    )
    
    print(f"{user_name} -> {role_name} (scope: {scope_type}/{scope_id}) - Status: {response.status_code}")
    if response.status_code not in [200, 201]:
        print(f"  Error: {response.text}")

print("-" * 40)
print("Euroschool role assignments complete!")

Assigning roles to Euroschool users...
----------------------------------------
Ammu -> STUDENT (scope: CLASS/4-C) - Status: 200
Priya -> SUBJECT_TEACHER (scope: CLASS/4-C) - Status: 200
Principal1 -> PRINCIPAL (scope: TENANT/None) - Status: 200
----------------------------------------
Euroschool role assignments complete!


### 4.2 Assign Roles to Ravishankarschool Users

In [70]:
# Assign roles to Ravishankarschool users

print(f"Assigning roles to Ravishankarschool users...")
print("-" * 40)

for user_name, user_info in ravishankar_user_data.items():
    role_name = user_info["role"]
    user_id = user_info["userId"]
    role_id = ravishankar_role_ids.get(role_name)
    
    if not role_id:
        print(f"{user_name} - Role {role_name} not found!")
        continue
    
    # Determine scope based on role
    if role_name == "PRINCIPAL":
        scope_type = "TENANT"
        scope_id = None
    elif role_name == "SUBJECT_TEACHER":
        scope_type = "CLASS"
        scope_id = "4-D"  # Ravi teaches Class 4-D
    else:  # STUDENT
        scope_type = "CLASS"
        scope_id = "4-D"  # Daksh is in Class 4-D
    
    payload = {
        "roleId": role_id,
        "scopeType": scope_type,
        "scopeId": scope_id,  # Must always be included, even when None
        "status": "ACTIVE"
    }
    
    response = requests.post(
        f"{ROLE_PERMISSION_URL}/tenants/{ravishankarschool_id}/users/{user_id}/roles",
        headers={"Content-Type": "application/json"},
        json=payload
    )
    
    print(f"{user_name} -> {role_name} (scope: {scope_type}/{scope_id}) - Status: {response.status_code}")
    if response.status_code not in [200, 201]:
        print(f"  Error: {response.text}")

print("-" * 40)
print("Ravishankarschool role assignments complete!")

Assigning roles to Ravishankarschool users...
----------------------------------------
Daksh -> STUDENT (scope: CLASS/4-D) - Status: 200
Ravi -> SUBJECT_TEACHER (scope: CLASS/4-D) - Status: 200
Principal2 -> PRINCIPAL (scope: TENANT/None) - Status: 200
----------------------------------------
Ravishankarschool role assignments complete!


---

## Step 5: Create Test Notes

### 5.1 Priya Creates Notes for Class 4-C (Euroschool)

In [71]:
# Priya (Euroschool Teacher) creates notes for her class

priya_token = euroschool_user_data.get("Priya", {}).get("accessToken")

notes_by_priya = [
    {
        "title": "Science: Introduction to Plants",
        "content": "Plants are living organisms that make their own food through photosynthesis.",
        "subject": "Science",
        "classId": "4-C"
    },
    {
        "title": "Social Science: Our Country India",
        "content": "India is a diverse country with many states and union territories.",
        "subject": "Social Science",
        "classId": "4-C"
    }
]

print("Priya creating notes for Euroschool Class 4-C...")
print("-" * 40)

euroschool_notes = []

for note in notes_by_priya:
    response = requests.post(
        f"{NOTES_URL}/notes",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {priya_token}",
            "X-Tenant-Id": str(euroschool_id)
        },
        json=note
    )
    
    if response.status_code in [200, 201]:
        note_data = response.json()
        euroschool_notes.append(note_data)
        created_resources["notes"].append({"tenant_id": euroschool_id, "note_id": note_data.get("id")})
        print(f"Created: {note['title']} (ID: {note_data.get('id')})")
    else:
        print(f"Failed: {note['title']} - {response.status_code}")
        print(f"  Error: {response.text}")

print("-" * 40)
print(f"Created {len(euroschool_notes)} note(s) for Euroschool")

Priya creating notes for Euroschool Class 4-C...
----------------------------------------
Created: Science: Introduction to Plants (ID: c2f49ed4-66d7-4755-9b00-aa60a66bbbd5)
Created: Social Science: Our Country India (ID: b27e512e-9a16-45b1-8a63-41f11f698595)
----------------------------------------
Created 2 note(s) for Euroschool


### 5.2 Ravi Creates Notes for Class 4-D (Ravishankarschool)

In [72]:
# Ravi (Ravishankarschool Teacher) creates notes for his class

ravi_token = ravishankar_user_data.get("Ravi", {}).get("accessToken")

notes_by_ravi = [
    {
        "title": "Science: Water Cycle",
        "content": "The water cycle describes how water evaporates, forms clouds, and returns as rain.",
        "subject": "Science",
        "classId": "4-D"
    },
    {
        "title": "Social Science: Indian Freedom Fighters",
        "content": "Many brave freedom fighters sacrificed their lives for India's independence.",
        "subject": "Social Science",
        "classId": "4-D"
    }
]

print("Ravi creating notes for Ravishankarschool Class 4-D...")
print("-" * 40)

ravishankar_notes = []

for note in notes_by_ravi:
    response = requests.post(
        f"{NOTES_URL}/notes",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {ravi_token}",
            "X-Tenant-Id": str(ravishankarschool_id)
        },
        json=note
    )
    
    if response.status_code in [200, 201]:
        note_data = response.json()
        ravishankar_notes.append(note_data)
        created_resources["notes"].append({"tenant_id": ravishankarschool_id, "note_id": note_data.get("id")})
        print(f"Created: {note['title']} (ID: {note_data.get('id')})")
    else:
        print(f"Failed: {note['title']} - {response.status_code}")
        print(f"  Error: {response.text}")

print("-" * 40)
print(f"Created {len(ravishankar_notes)} note(s) for Ravishankarschool")

Ravi creating notes for Ravishankarschool Class 4-D...
----------------------------------------
Created: Science: Water Cycle (ID: e4428c51-4841-4f3b-a97c-0bf666e010df)
Created: Social Science: Indian Freedom Fighters (ID: aecfa032-9dc0-492b-a30c-752faf9fa4dc)
----------------------------------------
Created 2 note(s) for Ravishankarschool


---

## Step 6: Test Access Scenarios

### Test 1: Ammu can see Euroschool notes

In [73]:
# Test: Ammu (Euroschool Student) tries to read notes

ammu_token = euroschool_user_data.get("Ammu", {}).get("accessToken")

print("Test 1: Ammu reading Euroschool notes...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {ammu_token}",
        "X-Tenant-Id": str(euroschool_id)
    }
)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    data = response.json()
    # Handle paginated response - Notes service uses 'items' key
    notes = data.get("items", []) if isinstance(data, dict) else data
    print(f"Ammu can see {len(notes)} note(s):")
    for note in notes:
        print(f"  - {note.get('title')}")
    print("\n[PASS] Ammu can access Euroschool notes")
else:
    print(f"Error: {response.text}")
    print("\n[FAIL] Ammu cannot access notes")

Test 1: Ammu reading Euroschool notes...
----------------------------------------
Status: 200
Ammu can see 2 note(s):
  - Social Science: Our Country India
  - Science: Introduction to Plants

[PASS] Ammu can access Euroschool notes


### Test 2: Daksh can see Ravishankarschool notes

In [74]:
# Test: Daksh (Ravishankarschool Student) tries to read notes

daksh_token = ravishankar_user_data.get("Daksh", {}).get("accessToken")

print("Test 2: Daksh reading Ravishankarschool notes...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {daksh_token}",
        "X-Tenant-Id": str(ravishankarschool_id)
    }
)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    data = response.json()
    # Handle paginated response - Notes service uses 'items' key
    notes = data.get("items", []) if isinstance(data, dict) else data
    print(f"Daksh can see {len(notes)} note(s):")
    for note in notes:
        print(f"  - {note.get('title')}")
    print("\n[PASS] Daksh can access Ravishankarschool notes")
else:
    print(f"Error: {response.text}")
    print("\n[FAIL] Daksh cannot access notes")

Test 2: Daksh reading Ravishankarschool notes...
----------------------------------------
Status: 200
Daksh can see 2 note(s):
  - Social Science: Indian Freedom Fighters
  - Science: Water Cycle

[PASS] Daksh can access Ravishankarschool notes


### Test 3: Ammu CANNOT see Ravishankarschool notes (Tenant Isolation)

In [75]:
# Test: Ammu (Euroschool Student) tries to read Ravishankarschool notes - SHOULD FAIL

ammu_token = euroschool_user_data.get("Ammu", {}).get("accessToken")

print("Test 3: Ammu trying to read Ravishankarschool notes (should fail)...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {ammu_token}",
        "X-Tenant-Id": str(ravishankarschool_id)  # Wrong tenant!
    }
)

print(f"Status: {response.status_code}")
if response.status_code in [401, 403]:
    print("Access denied (as expected)")
    print("\n[PASS] Tenant isolation working - Ammu cannot access other school's notes")
elif response.status_code == 200:
    data = response.json()
    # Handle paginated response - Notes service uses 'items' key
    notes = data.get("items", []) if isinstance(data, dict) else data
    if len(notes) == 0:
        print("Empty result (tenant isolation via empty response)")
        print("\n[PASS] Tenant isolation working")
    else:
        print(f"[FAIL] Ammu can see {len(notes)} notes from other school!")
else:
    print(f"Response: {response.text}")

Test 3: Ammu trying to read Ravishankarschool notes (should fail)...
----------------------------------------
Status: 200
[FAIL] Ammu can see 2 notes from other school!


### Test 4: Priya CANNOT see Ravishankarschool notes (Tenant Isolation)

In [76]:
# Test: Priya (Euroschool Teacher) tries to read Ravishankarschool notes - SHOULD FAIL

priya_token = euroschool_user_data.get("Priya", {}).get("accessToken")

print("Test 4: Priya trying to read Ravishankarschool notes (should fail)...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {priya_token}",
        "X-Tenant-Id": str(ravishankarschool_id)  # Wrong tenant!
    }
)

print(f"Status: {response.status_code}")
if response.status_code in [401, 403]:
    print("Access denied (as expected)")
    print("\n[PASS] Tenant isolation working - Priya cannot access other school's notes")
elif response.status_code == 200:
    data = response.json()
    # Handle paginated response - Notes service uses 'items' key
    notes = data.get("items", []) if isinstance(data, dict) else data
    if len(notes) == 0:
        print("Empty result (tenant isolation via empty response)")
        print("\n[PASS] Tenant isolation working")
    else:
        print(f"[FAIL] Priya can see {len(notes)} notes from other school!")
else:
    print(f"Response: {response.text}")

Test 4: Priya trying to read Ravishankarschool notes (should fail)...
----------------------------------------
Status: 200
[FAIL] Priya can see 2 notes from other school!


### Test 5: Principal1 can see ALL Euroschool notes

In [77]:
# Test: Principal1 (Euroschool) can see all notes in the school

principal1_token = euroschool_user_data.get("Principal1", {}).get("accessToken")

print("Test 5: Principal1 reading all Euroschool notes...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {principal1_token}",
        "X-Tenant-Id": str(euroschool_id)
    }
)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    data = response.json()
    # Handle paginated response - Notes service uses 'items' key
    notes = data.get("items", []) if isinstance(data, dict) else data
    print(f"Principal1 can see {len(notes)} note(s):")
    for note in notes:
        print(f"  - {note.get('title')}")
    print("\n[PASS] Principal has school-wide access")
else:
    print(f"Error: {response.text}")
    print("\n[FAIL] Principal cannot access notes")

Test 5: Principal1 reading all Euroschool notes...
----------------------------------------
Status: 200
Principal1 can see 2 note(s):
  - Social Science: Our Country India
  - Science: Introduction to Plants

[PASS] Principal has school-wide access


---

## Cleanup

### View Created Resources

In [78]:
# View all created resources
print("Created Resources Summary:")
print("=" * 40)
print(json.dumps(created_resources, indent=2))

Created Resources Summary:
{
  "tenants": [],
  "users": [
    {
      "tenant_id": "0f66209d-f36f-4567-b24a-3d32a31bce80",
      "user_id": "60f3c705-7fcb-43a3-b5ab-4b8cbfecbc43"
    },
    {
      "tenant_id": "0f66209d-f36f-4567-b24a-3d32a31bce80",
      "user_id": "cb23f11f-0294-4ccb-9eaa-77c8779461f2"
    },
    {
      "tenant_id": "0f66209d-f36f-4567-b24a-3d32a31bce80",
      "user_id": "6846ea99-d0fa-430f-a323-4775b67a5011"
    },
    {
      "tenant_id": "43e49388-a868-45ae-a45c-7de2f646871d",
      "user_id": "f13d09b7-4645-4fdb-9322-7cc6ecda9d0e"
    },
    {
      "tenant_id": "43e49388-a868-45ae-a45c-7de2f646871d",
      "user_id": "1b123b44-12d9-4102-a756-bdebb2ac1e12"
    },
    {
      "tenant_id": "43e49388-a868-45ae-a45c-7de2f646871d",
      "user_id": "f56af500-e940-4b0a-8a9f-5a2d50302f74"
    }
  ],
  "roles": [],
  "grants": [],
  "assignments": [],
  "notes": [
    {
      "tenant_id": "0f66209d-f36f-4567-b24a-3d32a31bce80",
      "note_id": "c2f49ed4-66d7-4755-9b

### Delete Notes

In [79]:
# Delete all created notes
print("Deleting notes...")
print("-" * 40)

for note_info in created_resources.get("notes", []):
    note_id = note_info.get("note_id")
    tenant_id = note_info.get("tenant_id")
    
    response = requests.delete(
        f"{NOTES_URL}/notes/{note_id}",
        headers={"X-Tenant-Id": str(tenant_id)}
    )
    print(f"Deleting note {note_id} - Status: {response.status_code}")

print("-" * 40)
print("Notes cleanup complete!")

Deleting notes...
----------------------------------------
Deleting note c2f49ed4-66d7-4755-9b00-aa60a66bbbd5 - Status: 204
Deleting note b27e512e-9a16-45b1-8a63-41f11f698595 - Status: 204
Deleting note e4428c51-4841-4f3b-a97c-0bf666e010df - Status: 204
Deleting note aecfa032-9dc0-492b-a30c-752faf9fa4dc - Status: 204
----------------------------------------
Notes cleanup complete!


### Delete Role Assignments

In [80]:
# Delete role assignments
# Note: This requires fetching assignments first, then deleting

print("Deleting role assignments...")
print("-" * 40)

for user_info in created_resources.get("users", []):
    user_id = user_info.get("user_id")
    tenant_id = user_info.get("tenant_id")
    
    # Get assignments for user
    response = requests.get(f"{ROLE_PERMISSION_URL}/tenants/{tenant_id}/users/{user_id}/roles")
    if response.status_code == 200:
        assignments = response.json()
        for assignment in assignments:
            assign_id = assignment.get("id")
            del_response = requests.delete(
                f"{ROLE_PERMISSION_URL}/tenants/{tenant_id}/users/{user_id}/roles/{assign_id}"
            )
            print(f"Deleting assignment {assign_id} for user {user_id} - Status: {del_response.status_code}")

print("-" * 40)
print("Role assignments cleanup complete!")

Deleting role assignments...
----------------------------------------
Deleting assignment 7 for user 60f3c705-7fcb-43a3-b5ab-4b8cbfecbc43 - Status: 200
Deleting assignment 8 for user cb23f11f-0294-4ccb-9eaa-77c8779461f2 - Status: 200
Deleting assignment 9 for user 6846ea99-d0fa-430f-a323-4775b67a5011 - Status: 200
Deleting assignment 10 for user f13d09b7-4645-4fdb-9322-7cc6ecda9d0e - Status: 200
Deleting assignment 11 for user 1b123b44-12d9-4102-a756-bdebb2ac1e12 - Status: 200
Deleting assignment 12 for user f56af500-e940-4b0a-8a9f-5a2d50302f74 - Status: 200
----------------------------------------
Role assignments cleanup complete!


### Delete Roles

In [81]:
# Delete roles (grants are deleted with roles)
print("Deleting roles...")
print("-" * 40)

for role_info in created_resources.get("roles", []):
    role_id = role_info.get("role_id")
    tenant_id = role_info.get("tenant_id")
    
    response = requests.delete(f"{ROLE_PERMISSION_URL}/tenants/{tenant_id}/roles/{role_id}")
    print(f"Deleting role {role_id} from tenant {tenant_id} - Status: {response.status_code}")

print("-" * 40)
print("Roles cleanup complete!")

Deleting roles...
----------------------------------------
----------------------------------------
Roles cleanup complete!


### Delete Users (Optional)

Note: User deletion depends on auth service API availability.

In [ ]:
# Delete users (if auth service supports it)
print("Deleting users...")
print("-" * 40)

for user_info in created_resources.get("users", []):
    user_id = user_info.get("user_id")
    tenant_id = user_info.get("tenant_id")
    
    # Try to delete via auth service
    response = requests.delete(f"{AUTH_URL}/auth/users/{user_id}")
    print(f"Deleting user {user_id} - Status: {response.status_code}")
    if response.status_code not in [200, 204]:
        print(f"  Note: {response.text}")

print("-" * 40)
print("Users cleanup complete!")

## Step 7: Publish Workflow Tests

These tests verify that:
1. Students cannot see DRAFT notes
2. Teachers can see their own DRAFT notes
3. After publishing, students can see the notes

### Setup: Create Users and Tenant for Publish Tests

In [ ]:
# Setup for publish workflow tests
# We'll create fresh users for this test to avoid conflicts

print("Setting up Publish Workflow Tests...")
print("=" * 50)

# Create a test teacher
teacher_signup = requests.post(
    f"{AUTH_URL}/auth/signup",
    json={
        "email": f"teacher_publish_test_{uuid.uuid4().hex[:8]}@euroschool.edu",
        "password": "Test@123",
        "tenantId": str(euroschool_id),
        "displayName": "Test Teacher"
    }
)
teacher_data = teacher_signup.json()
teacher_id = teacher_data.get("userId")
teacher_token = teacher_data.get("accessToken")
print(f"Created Teacher: {teacher_id}")

# Create a test student
student_signup = requests.post(
    f"{AUTH_URL}/auth/signup",
    json={
        "email": f"student_publish_test_{uuid.uuid4().hex[:8]}@euroschool.edu",
        "password": "Test@123",
        "tenantId": str(euroschool_id),
        "displayName": "Test Student"
    }
)
student_data = student_signup.json()
student_id = student_data.get("userId")
student_token = student_data.get("accessToken")
print(f"Created Student: {student_id}")

print("=" * 50)
print("Setup complete!")

### Test 6: Teacher Creates a DRAFT Note

In [ ]:
# Test 6: Teacher creates a DRAFT note
print("Test 6: Teacher creates a DRAFT note...")
print("-" * 40)

draft_note = {
    "title": "Draft: Advanced Mathematics",
    "summary": "This is a draft note that students should NOT see",
    "contentMd": "# Advanced Mathematics\n\nThis content is still being prepared...",
    "createdBy": str(teacher_id)
}

response = requests.post(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {teacher_token}",
        "X-Tenant-Id": str(euroschool_id)
    },
    json=draft_note
)

if response.status_code in [200, 201]:
    draft_note_data = response.json()
    draft_note_id = draft_note_data.get("id")
    draft_note_status = draft_note_data.get("status")
    print(f"Created Note ID: {draft_note_id}")
    print(f"Note Status: {draft_note_status}")
    print(f"Note Title: {draft_note_data.get('title')}")
    
    if draft_note_status == "DRAFT":
        print("\n[PASS] Note created with DRAFT status")
    else:
        print(f"\n[FAIL] Expected DRAFT status, got: {draft_note_status}")
else:
    print(f"Error creating note: {response.status_code}")
    print(response.text)
    draft_note_id = None

### Test 7: Student CANNOT See DRAFT Notes

In [ ]:
# Test 7: Student tries to see notes - should NOT see the DRAFT note
print("Test 7: Student tries to list notes (should NOT see DRAFT notes)...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {student_token}",
        "X-Tenant-Id": str(euroschool_id)
    }
)

print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", []) if isinstance(data, dict) else data
    
    # Check if the draft note is in the list
    draft_note_visible = any(note.get("id") == str(draft_note_id) for note in notes)
    
    print(f"Student can see {len(notes)} note(s)")
    for note in notes:
        print(f"  - {note.get('title')} (Status: {note.get('status')})")
    
    if not draft_note_visible:
        print("\n[PASS] Student cannot see the DRAFT note - publish workflow working!")
    else:
        print("\n[FAIL] Student can see the DRAFT note - publish workflow NOT working!")
else:
    print(f"Error: {response.text}")

### Test 8: Teacher CAN See Their Own DRAFT Notes

In [ ]:
# Test 8: Teacher can see their own DRAFT notes (by filtering with createdBy)
print("Test 8: Teacher lists their own notes (should see DRAFT notes)...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {teacher_token}",
        "X-Tenant-Id": str(euroschool_id)
    },
    params={
        "createdBy": str(teacher_id)  # Filter by creator to see own drafts
    }
)

print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", []) if isinstance(data, dict) else data
    
    # Check if the draft note is in the list
    draft_note_visible = any(note.get("id") == str(draft_note_id) for note in notes)
    
    print(f"Teacher can see {len(notes)} note(s)")
    for note in notes:
        print(f"  - {note.get('title')} (Status: {note.get('status')})")
    
    if draft_note_visible:
        print("\n[PASS] Teacher can see their own DRAFT note")
    else:
        print("\n[FAIL] Teacher cannot see their own DRAFT note")
else:
    print(f"Error: {response.text}")

### Test 9: Teacher Publishes the Note

In [ ]:
# Test 9: Teacher publishes the note
print("Test 9: Teacher publishes the note...")
print("-" * 40)

if draft_note_id:
    response = requests.post(
        f"{NOTES_URL}/notes/{draft_note_id}/publish",
        headers={
            "Authorization": f"Bearer {teacher_token}",
            "X-Tenant-Id": str(euroschool_id)
        },
        json={
            "publishedBy": str(teacher_id)
        }
    )
    
    print(f"Status: {response.status_code}")
    
    if response.status_code == 200:
        published_note = response.json()
        print(f"Note ID: {published_note.get('id')}")
        print(f"Note Status: {published_note.get('status')}")
        print(f"Published At: {published_note.get('publishedAt')}")
        
        if published_note.get("status") == "PUBLISHED":
            print("\n[PASS] Note successfully published!")
        else:
            print(f"\n[FAIL] Expected PUBLISHED status, got: {published_note.get('status')}")
    else:
        print(f"Error: {response.text}")
else:
    print("Skipping - no draft note was created")

### Test 10: Student CAN Now See the PUBLISHED Note

In [ ]:
# Test 10: Student can now see the PUBLISHED note
print("Test 10: Student lists notes (should NOW see the PUBLISHED note)...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {student_token}",
        "X-Tenant-Id": str(euroschool_id)
    }
)

print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", []) if isinstance(data, dict) else data
    
    # Check if the (now published) note is in the list
    published_note_visible = any(note.get("id") == str(draft_note_id) for note in notes)
    
    print(f"Student can see {len(notes)} note(s)")
    for note in notes:
        print(f"  - {note.get('title')} (Status: {note.get('status')})")
    
    if published_note_visible:
        print("\n[PASS] Student can now see the PUBLISHED note - workflow complete!")
    else:
        print("\n[FAIL] Student still cannot see the published note")
else:
    print(f"Error: {response.text}")

### Cleanup: Delete Test Resources

In [ ]:
# Cleanup publish workflow test resources
print("Cleaning up publish workflow test resources...")
print("-" * 40)

# Delete the test note
if draft_note_id:
    response = requests.delete(
        f"{NOTES_URL}/notes/{draft_note_id}",
        headers={"X-Tenant-Id": str(euroschool_id)}
    )
    print(f"Deleted note {draft_note_id} - Status: {response.status_code}")

# Delete test users
for user_id in [teacher_id, student_id]:
    if user_id:
        response = requests.delete(f"{AUTH_URL}/auth/users/{user_id}")
        print(f"Deleted user {user_id} - Status: {response.status_code}")

print("-" * 40)
print("Cleanup complete!")

### Delete Tenants (Optional)

Note: Be careful - this will remove the schools!

In [82]:
# Delete tenants
print("Deleting tenants...")
print("-" * 40)

for tenant_id in created_resources.get("tenants", []):
    response = requests.delete(f"{TENANT_URL}/v1/tenants/{tenant_id}")
    print(f"Deleting tenant {tenant_id} - Status: {response.status_code}")

print("-" * 40)
print("Tenants cleanup complete!")

Deleting tenants...
----------------------------------------
----------------------------------------
Tenants cleanup complete!


# User student must not be able to view notes created by teacher untill teacher publishes them to students 